# 15. MCP Client Development

This notebook demonstrates how to use the **FastMCP 3** client against the
Atlas Curated MCP Servers endpoint, covering discovery, invocation, error
handling, and in-process experimental tool authoring.

The MCP endpoint is available when `MCP_SERVERS_SOURCE=container` (disabled
by default). When disabled, the cells below handle the unavailable-state
gracefully.

In [ ]:
import os

from fastmcp import Client

def _mcp_exception_text(exc):
    parts = []
    seen = set()
    current = exc
    while current is not None and id(current) not in seen:
        seen.add(id(current))
        parts.append(f"{type(current).__name__}: {current}")
        current = current.__cause__ or current.__context__
    return " | ".join(parts)

def _mcp_error_category(exc):
    detail = _mcp_exception_text(exc).lower()
    if any(token in detail for token in ("http 401", "http 403", "401 unauthorized", "403 forbidden", "status 401", "status 403", "client error '401", "client error '403", "authentication failed", "permission denied")):
        return "authentication"
    if any(token in detail for token in ("timeout", "timed out")):
        return "timeout"
    if any(token in detail for token in ("connecterror", "connectionerror", "connection refused", "unreachable", "name resolution", "nodename nor servname", "dns")):
        return "unreachable"
    if any(token in detail for token in ("http 500", "http 502", "http 503", "http 504", "server error", "service unavailable")):
        return "server"
    return "client/server"

def _report_mcp_error(stage, exc):
    category = _mcp_error_category(exc)
    print(f"MCP {stage} failed ({category}) [{type(exc).__name__}].")

def _is_missing_tool_error(exc):
    detail = _mcp_exception_text(exc).lower()
    return any(token in detail for token in ("unknown tool", "tool not found", "does not exist"))

MCP_TOOLS = None
MCP_SERVERS_URL = os.environ.get("MCP_SERVERS_URL", "")
if not MCP_SERVERS_URL:
    print("⚠ MCP_SERVERS_URL is not set — MCP_SERVERS_SOURCE is likely disabled.")
    print("  Enable with: ./start.sh --mcp-servers-source container")
else:
    print("MCP endpoint is configured (value hidden).")

## Discovery

List the tools the curated MCP server exposes, including their input schemas.

In [ ]:
if MCP_SERVERS_URL:
    try:
        async with Client(MCP_SERVERS_URL) as client:
            tools = await client.list_tools()
        MCP_TOOLS = {tool.name: tool for tool in tools}
        for tool in tools:
            print(f"  {tool.name}: {tool.description or '(no description)'}")
            if tool.inputSchema:
                print(f"    schema: {tool.inputSchema}")
    except Exception as exc:
        _report_mcp_error("discovery", exc)
else:
    print("(MCP disabled — skipping discovery)")

## Invocation

Call the bounded web-search tool with fixed input (`Atlas`, limit 1). Live
results depend on SearXNG; the fixed request keeps the smoke bounded and
avoids sending placeholder text to a database query tool.

In [ ]:
if not MCP_SERVERS_URL:
    print("(MCP disabled — skipping invocation)")
elif MCP_TOOLS is None:
    print("(MCP invocation skipped — discovery did not complete)")
else:
    tool = MCP_TOOLS.get("searxng_web_search")
    if tool is None:
        print("MCP target tool missing: searxng_web_search")
    else:
        args = {"query": "Atlas", "limit": 1}
        try:
            async with Client(MCP_SERVERS_URL) as client:
                result = await client.call_tool(tool.name, args)
            print(f"{tool.name}({args}) -> {result}")
        except Exception as exc:
            _report_mcp_error("tool invocation", exc)



## Error Handling

Handle the common failure modes: disabled service, authentication, timeout,
and tool errors.

In [ ]:
if not MCP_SERVERS_URL:
    print("(MCP disabled — skipping error handling)")
elif MCP_TOOLS is None:
    print("(MCP error-handling probe skipped — discovery did not complete)")
else:
    try:
        async with Client(MCP_SERVERS_URL) as client:
            await client.call_tool("nonexistent_tool", {})
    except Exception as exc:
        if _is_missing_tool_error(exc):
            print(f"Caught expected missing-tool error ({type(exc).__name__}).")
        else:
            _report_mcp_error("error-handling probe", exc)

## In-Process Experimental Tools

FastMCP lets you define and test a tool **in-process** — no network service
needed. This is useful for prototyping new MCP tools before publishing them.

In [ ]:
from fastmcp import FastMCP

# Define a tiny experimental tool in-process.
app = FastMCP("experimental")

@app.tool()
def greet(name: str) -> str:
    """Greet someone by name."""
    return f"Hello, {name}!"

# Test it via the in-memory transport (no network).
async with Client(app) as client:
    result = await client.call_tool("greet", {"name": "Atlas"})
    print(result)